# Figure S1: threshold burden by home

Percentage of each home's valid observations beyond the descriptive thresholds of Methods 2.3, for all 23 homes.
The values are those of Table S1 and Results 3.2; the figure shows them in the Home ID order of Figure 1.

**Step:** 16. Article figure (run after step 04 if Table S1 should be cross-checked; otherwise it reads the data directly).

**Article scope:** the main article and Supplementary Material.

**Inputs:** `data/hourly_primary_23.csv.gz` and `data/daily_home.csv` (prepared files in this run take precedence over supplied inputs in `data/`), through `load_hourly` and `load_daily` in `00_shared_functions.ipynb`.

**Outputs:** `FigS1_threshold_burden.png` (900 dpi) and `.pdf` in the output `figures/` folder, and the plotted values in `tables/FigS1_threshold_burden_values.csv`.

**Rules (identical to step 04):** PM2.5, percentage of valid daily means above 15 µg/m3; temperature, percentage of valid hours at or below 18 °C;
RH, percentage of valid hours at or above 60%; CO2, percentage of valid hours at or above 1,000 ppm. Missing observations are excluded from numerators and denominators.

**How to use this notebook:** select the Python environment described in `README.md`, restart the kernel,
and run the cells from top to bottom. Edit the configuration cell before running.
The checks below stop execution when the current data no longer match the manuscript specification.

## Locate the code folder and load the shared functions

The repository is found by searching this notebook's folder and its parent folders, so Jupyter can be started anywhere inside the repository. Set `INDOOR_AQ_CODE` only when it starts elsewhere.
It loads only the cells tagged `library` in `00_shared_functions.ipynb` (its demonstration cell is not executed).
Each notebook has its own imports and configuration, so it never depends on another notebook's kernel state.

In [ ]:
# Setup: find the code folder and load the shared functions.
from pathlib import Path
import json
import os
import sys
import types


def locate_code_directory():
    """Locate this repository (the folder with 00_shared_functions.ipynb) from the working directory or INDOOR_AQ_CODE."""
    configured = os.environ.get("INDOOR_AQ_CODE")
    if configured:
        candidates = [Path(configured).expanduser().resolve()]
    else:
        candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if ((candidate / "00_shared_functions.ipynb").is_file()
                and (candidate / "5_did_cits").is_dir()):
            return candidate.resolve()
    raise FileNotFoundError("Start Jupyter in the repository folder, or set INDOOR_AQ_CODE to that folder.")

CODE = locate_code_directory()

# Load the cells tagged "library" from 00_shared_functions.ipynb as an in-memory
# module named `shared`, so that `from shared import ...` works below.
# No helper .py file is generated.
_library_path = CODE / "00_shared_functions.ipynb"
_library = types.ModuleType("shared")
_library.__file__ = str(_library_path)
_library.__dict__["CODE"] = CODE
for _cell in json.loads(_library_path.read_text(encoding="utf-8"))["cells"]:
    if _cell["cell_type"] == "code" and "library" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), str(_library_path), "exec"), _library.__dict__)
sys.modules["shared"] = _library

from shared import DATA, output_root
print(f"Manuscript code: {CODE}")

## Configure figure paths and resolution

Only presentation and output settings belong here. Keep the manuscript data and event specification fixed when reproducing the final figure.

In [ ]:
# Paths are shared by Python and R. Defaults: data/ for supplied inputs,
# results/ for this run. Set INDOOR_AQ_DATA and INDOOR_AQ_OUTPUT before
# starting Jupyter and R; relative paths are resolved against the repository.
OUTPUT = os.environ.get("INDOOR_AQ_OUTPUT") or None

# Set a lower DPI only for an explicit draft; manuscript exports use 900.
FIGURE_DPI = int(os.environ.get("INDOOR_AQ_FIGURE_DPI", "900"))
os.environ["INDOOR_AQ_OUTPUT"] = str(output_root(OUTPUT))
os.environ["INDOOR_AQ_FIGURE_DPI"] = str(FIGURE_DPI)
# All event-result figures use this run's results folder.


## Load plotting libraries and define the figure settings

The four panels follow the order and Home ID axis of Figure 1. Bars beyond the 20% reference line are drawn darker, because Results 3.2 counts the homes with more than 20% of valid data beyond each threshold.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import font_manager

from shared import CODE, OUTCOMES, crosswalk, load_daily, load_hourly, output_root

# ---- Output ----------------------------------------------------------------
OUTPUT = output_root()
OUTPUT_DIR = OUTPUT / 'figures'
OUTPUT_STEM = 'FigS1_threshold_burden'
EXPORT_FORMATS = ('png', 'pdf')
DPI = int(os.environ.get('INDOOR_AQ_FIGURE_DPI', '900'))
VALUES_PATH = OUTPUT / 'tables' / 'FigS1_threshold_burden_values.csv'

# ---- Font and library version used for the manuscript figure ----------------
FONT_FAMILY = 'Liberation Sans'
FALLBACK_FONT = 'Arial'  # same character widths as Liberation Sans, so the layout is unchanged
FONT_SEARCH_DIRS = [
    Path(os.environ.get('WINDIR', r'C:\Windows')) / 'Fonts',
    Path(os.environ.get('LOCALAPPDATA', '')) / 'Microsoft' / 'Windows' / 'Fonts',
    Path(r'C:\Program Files\LibreOffice\share\fonts\truetype'),
    # Optional: a folder with LiberationSans-*.ttf can be supplied through INDOOR_AQ_FONT_DIR.
    *([Path(os.environ['INDOOR_AQ_FONT_DIR'])] if os.environ.get('INDOOR_AQ_FONT_DIR') else []),
]
EXPECTED_MATPLOTLIB = '3.11'  # other releases may shift text by a sub-pixel

# ---- Thresholds (Methods 2.3) ----------------------------------------------
# (operator, threshold, observation level). PM2.5 uses valid daily means;
# the other metrics use valid hourly values.
THRESHOLDS = {
    'temp': ('le', 18, 'hourly'),
    'rh': ('ge', 60, 'hourly'),
    'pm25': ('gt', 15, 'daily'),
    'co2': ('ge', 1000, 'hourly'),
}
# Number of homes with more than 20% of valid data beyond each threshold (Results 3.2).
EXPECTED_COUNTS = {'temp': 10, 'rh': 16, 'pm25': 20, 'co2': 8}
REFERENCE_PERCENT = 20

# ---- Panels (top to bottom, as in Figure 1) ---------------------------------
VARIABLES = ['temp', 'rh', 'pm25', 'co2']
PANEL_LETTERS = 'abcd'
Y_LABELS = {
    'temp': 'Temperature ≤18 °C\n(% of valid hours)',
    'rh': 'RH ≥60%\n(% of valid hours)',
    'pm25': 'PM$_{2.5}$ >15 µg/m$^3$\n(% of valid days)',
    'co2': 'CO$_2$ ≥1,000 ppm\n(% of valid hours)',
}

# ---- Colors -----------------------------------------------------------------
BAR_ABOVE = '#6F6A64'      # more than 20% of valid data beyond the threshold
BAR_BELOW = '#D3CEC7'      # 20% or less
REFERENCE_COLOR = '#2B2825'
COUNT_COLOR = '#3A3632'

# ---- Geometry -----------------------------------------------------------------
# Home i (Home ID i + 1) sits at x = i, as in Figure 1.
BAR_WIDTH = 0.68
X_LIMITS = (-0.65, 22.65)
Y_LIMITS = (0, 104)
Y_TICKS = [0, 20, 40, 60, 80, 100]

# ---- Figure layout (inches and figure fractions) -----------------------------
FIGURE_SIZE = (6.3, 5.9)
SUBPLOT_LAYOUT = dict(left=0.135, right=0.99, bottom=0.075, top=0.965, hspace=0.34)

## Check matplotlib version

Report whether Matplotlib matches the release used for the manuscript. Fonts and text placement may differ across releases even when the underlying data and geometry agree.

In [ ]:
def check_matplotlib_version(expected=EXPECTED_MATPLOTLIB):
    """Warn if matplotlib differs from the release used for the manuscript figure."""
    if not matplotlib.__version__.startswith(expected + '.'):
        warnings.warn(
            f'The manuscript figure was rendered with matplotlib {expected}.x; '
            f'this kernel has {matplotlib.__version__}.')

## Resolve font

Prefer the manuscript font, register an available local font file if needed, and use the stated fallback when unavailable.

In [ ]:
def resolve_font(family=FONT_FAMILY, search_dirs=FONT_SEARCH_DIRS, fallback=FALLBACK_FONT):
    """Return a font family that matplotlib can use, preferring ``family``."""
    def available():
        return {font.name for font in font_manager.fontManager.ttflist}

    if family in available():
        return family

    pattern = family.replace(' ', '') + '-*.ttf'
    for directory in search_dirs:
        for path in sorted(Path(directory).glob(pattern)):
            font_manager.fontManager.addfont(str(path))

    if family in available():
        return family
    warnings.warn(f'{family} not found; using {fallback}. Glyph shapes will differ slightly.')
    return fallback

## Apply style

Reset Matplotlib defaults before applying the figure's font, line, and export settings.

In [ ]:
def apply_style(font_family):
    """Reset matplotlib to its defaults, then apply the Fig. S1 style."""
    plt.rcdefaults()
    plt.rcParams.update({
        'font.family': font_family,
        'font.size': 8,
        'axes.labelsize': 8,
        'xtick.labelsize': 7.5,
        'ytick.labelsize': 7.5,
        'axes.linewidth': 0.6,
        'xtick.major.width': 0.6,
        'ytick.major.width': 0.6,
        'xtick.major.size': 2.5,
        'ytick.major.size': 2.5,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'mathtext.default': 'regular',  # sub/superscripts in the text font, not italic math
        'savefig.facecolor': 'white',
        'pdf.fonttype': 42,             # embed TrueType fonts so PDF text stays editable
    })

## Load observations

Read the hourly and daily data through the shared loaders, which resolve every home through its sensor ID. Days with fewer than 18 valid hours are already missing in the daily data; nothing is filled.

In [ ]:
def load_observations(output=OUTPUT):
    """Hourly values and valid daily means of the 23 homes, labelled with Home IDs."""
    hourly = load_hourly(output)
    daily = load_daily(output)
    daily['home_id'] = daily['sensor_id'].map(crosswalk().set_index('sensor_id')['home_id'])
    assert daily['home_id'].notna().all()
    return hourly, daily

## Calculate threshold burdens

For each home and metric, divide the number of valid observations beyond the threshold by the number of valid observations. This is the rule used for Table S1 in step 04.

In [ ]:
def threshold_burdens(hourly, daily, thresholds=THRESHOLDS):
    """Percentage of valid observations beyond each threshold: one row per Home ID, one column per metric."""
    compare = {'le': np.less_equal, 'ge': np.greater_equal, 'gt': np.greater}
    columns = {}
    for variable, (operator, threshold, level) in thresholds.items():
        if level == 'daily':
            values = daily.loc[daily['variable'].eq(variable) & daily['y'].notna(), ['home_id', 'y']]
            values = values.rename(columns={'y': 'value'})
        else:
            values = hourly.loc[hourly['variable'].eq(variable) & hourly['value'].notna(), ['home_id', 'value']]
        beyond = compare[operator](values['value'], threshold)
        columns[variable] = 100 * beyond.groupby(values['home_id']).mean()
    burdens = pd.DataFrame(columns).sort_index()
    burdens.index.name = 'home_id'
    assert burdens.index.tolist() == list(range(1, 24)) and burdens.notna().all().all()
    return burdens

## Check the burdens against the manuscript

The counts of homes above 20% must match Results 3.2. When step 04 has written Table S1 in the output folder, every plotted value must equal its Table S1 value.

In [ ]:
def check_burdens(burdens, output=OUTPUT):
    """Stop if the burdens disagree with Results 3.2 or with a rebuilt Table S1."""
    counts = {v: int((burdens[v] > REFERENCE_PERCENT).sum()) for v in VARIABLES}
    assert counts == EXPECTED_COUNTS, counts
    table_s1 = Path(output) / 'tables' / 'Table_S1.csv'
    if table_s1.is_file():
        s1 = pd.read_csv(table_s1).set_index('Home').sort_index()
        for v in VARIABLES:
            assert np.allclose(burdens[v].to_numpy(), s1[v + '_burden_pct'].to_numpy(), rtol=0, atol=1e-9), v
        return 'counts and all 92 values match Table S1'
    return 'counts match Results 3.2 (Table S1 not rebuilt in this output folder)'


## Draw bars

One bar per home in Home ID order. Bars beyond the 20% reference are darker; the reference line itself carries the same information, so the shade is not the only cue.

In [ ]:
def draw_bars(ax, values):
    """Draw one bar per home; ``values`` is indexed by Home ID 1-23."""
    x = np.arange(len(values))
    colors = [BAR_ABOVE if v > REFERENCE_PERCENT else BAR_BELOW for v in values]
    ax.bar(x, values.to_numpy(), width=BAR_WIDTH, color=colors, lw=0, zorder=2)

## Add the reference line and the count of homes

Mark 20% with a dashed line and state how many homes exceed it, as in Results 3.2.

In [ ]:
def add_reference_line(ax):
    """Dashed line at the 20% reference used in Results 3.2."""
    ax.axhline(REFERENCE_PERCENT, color=REFERENCE_COLOR, ls=(0, (4, 2.5)), lw=0.7, zorder=3)


def add_count(ax, values):
    """Write 'n of 23 homes >20%' above the upper right corner of the panel."""
    n = int((values > REFERENCE_PERCENT).sum())
    ax.annotate(f'{n} of {len(values)} homes >{REFERENCE_PERCENT}%', xy=(1, 1), xycoords='axes fraction',
                xytext=(0, 3), textcoords='offset points', ha='right', va='bottom',
                fontsize=7.5, color=COUNT_COLOR)

## Format panel

Apply the Home ID axis of Figure 1, the percentage axis, the grid, and the panel letter.

In [ ]:
def format_panel(ax, variable, letter, home_ids, show_x_labels=True):
    """Home ID axis (labels only on the bottom panel), y label, grid and panel letter."""
    ax.set_xticks(range(len(home_ids)), [str(home_id) for home_id in home_ids])
    ax.set_xlim(*X_LIMITS)
    if show_x_labels:
        ax.set_xlabel('Home ID', labelpad=3)
    else:
        ax.tick_params(axis='x', labelbottom=False)
    ax.set_ylim(*Y_LIMITS)
    ax.set_yticks(Y_TICKS)
    ax.set_ylabel(Y_LABELS[variable], labelpad=6)
    ax.grid(axis='y', color='#dddddd', lw=0.5, alpha=0.8)
    ax.set_axisbelow(True)
    ax.annotate(f'{letter})', xy=(0, 1), xycoords='axes fraction',
                xytext=(-46, 2), textcoords='offset points',
                fontsize=9, fontweight='bold', va='bottom')

## Build figure

Assemble the four stacked panels from the checked burdens. This function contains presentation logic only.

In [ ]:
def build_figure(burdens):
    """Build the four stacked panels of Fig. S1 and return the figure."""
    fig, axes = plt.subplots(len(VARIABLES), 1, figsize=FIGURE_SIZE)
    fig.subplots_adjust(**SUBPLOT_LAYOUT)
    home_ids = burdens.index.tolist()
    for ax, variable, letter in zip(axes, VARIABLES, PANEL_LETTERS):
        draw_bars(ax, burdens[variable])
        add_reference_line(ax)
        add_count(ax, burdens[variable])
        format_panel(ax, variable, letter, home_ids, show_x_labels=(variable == VARIABLES[-1]))
    return fig

## Save figure

PNG uses the configured resolution (900 dpi by default); PDF keeps vector elements. The output directory is separate from the manuscript source figures.

In [ ]:
def save_figure(fig, output_dir=OUTPUT_DIR, stem=OUTPUT_STEM,
                formats=EXPORT_FORMATS, dpi=DPI):
    """Save ``fig`` once per format and return the written paths."""
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for ext in formats:
        path = output_dir / f'{stem}.{ext}'
        fig.savefig(path, dpi=dpi)
        paths.append(path)
    return paths

## Build and export the figure

Compute and check the burdens, save the plotted values, then write the figure and print its paths.

In [ ]:
check_matplotlib_version()
apply_style(resolve_font())

hourly, daily = load_observations()
burdens = threshold_burdens(hourly, daily)
print(check_burdens(burdens))
burdens.round(6).to_csv(VALUES_PATH)
print(f'Plotted values: {VALUES_PATH}')

fig = build_figure(burdens)
for path in save_figure(fig):
    print(f'Saved {path}')

## Display a compact notebook preview

The preview uses a smaller PNG rendered from the same figure object. The saved publication PNG remains at the configured 900 dpi.

In [ ]:
import io
from IPython.display import Image, display
_preview = io.BytesIO()
fig.savefig(_preview, format="png", dpi=110, bbox_inches="tight")
display(Image(data=_preview.getvalue()))
plt.close(fig)